In [1]:
import sys

print("Python version:", sys.version)
print("Python location:", sys.executable)

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python location: c:\Users\ausu\Desktop\saudi-tech-research\.venv311\Scripts\python.exe


In [2]:
import json
from pathlib import Path
from urllib.request import urlopen

# Locate the project folder.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

raw_dir = project_dir / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

url = (
    "https://"
    "data.ksu.edu.sa/sites/data.ksu.edu.sa/files/users/user976/"
    "KSU-DMO-OD-DATASET-SciResarch-Publications-2025.json"
)

# Download and check that the content is valid JSON.
with urlopen(url, timeout=90) as response:
    raw_bytes = response.read()

ksu_data = json.loads(raw_bytes.decode("utf-8-sig"))

# Preserve the original downloaded content.
file_path = raw_dir / "ksu_publications_2025.json"
file_path.write_bytes(raw_bytes)

print("Saved to:", file_path)
print("JSON structure:", type(ksu_data).__name__)
print("Preview:")
print(json.dumps(ksu_data, ensure_ascii=False, indent=2)[:2500])

JSONDecodeError: Invalid \escape: line 3368 column 1428 (char 696496)

In [3]:
# Preserve the downloaded bytes exactly as received.
original_path = raw_dir / "ksu_publications_2025_original.json"
original_path.write_bytes(raw_bytes)

text = raw_bytes.decode("utf-8-sig")

try:
    ksu_data = json.loads(text)
except json.JSONDecodeError as error:
    print("Problem:", error.msg)
    print("Line:", error.lineno, "| Column:", error.colno)

    start = max(0, error.pos - 150)
    end = min(len(text), error.pos + 150)

    print("\nText around the problem:")
    print(repr(text[start:end]))

Problem: Invalid \escape
Line: 3368 | Column: 1428

Text around the problem:
'onts} \\\\usepackage{amssymb} \\\\usepackage{amsbsy} \\\\usepackage{mathrsfs} \\\\usepackage{upgreek} \\\\setlength{\\\\oddsidemargin}{-69pt} \\\\begin{document}$$F\\\text {-score}$$\\\\end{document} of 72.78%. While generative performance reached BERT-F\\\\documentclass[12pt]{minimal} \\\\usepackage{amsmath} \\\\usepackag'


In [4]:
try:
    ksu_data = json.loads(text)
except json.JSONDecodeError as error:
    print("Characters around the error:")
    for position in range(max(0, error.pos - 5), error.pos + 12):
        character = text[position]
        print(position, repr(character), f"U+{ord(character):04X}")

Characters around the error:
696491 't' U+0074
696492 '}' U+007D
696493 '$' U+0024
696494 '$' U+0024
696495 'F' U+0046
696496 '\\' U+005C
696497 '\t' U+0009
696498 'e' U+0065
696499 'x' U+0078
696500 't' U+0074
696501 ' ' U+0020
696502 '{' U+007B
696503 '-' U+002D
696504 's' U+0073
696505 'c' U+0063
696506 'o' U+006F
696507 'r' U+0072


In [ ]:
# Create a working copy.
repaired_text = text

position = 696496
fragment = repaired_text[position:position + 2]

# Verify the exact characters before changing anything.
assert fragment == "\\" + "\t", "Characters differ; stop and inspect."



# Encode the backslash and tab correctly for JSON.
replacement = json.dumps(fragment)[1:-1]


repaired_text = (
    repaired_text[:position]
    + replacement
    + repaired_text[position + 2:]
)


try:
    ksu_data = json.loads(repaired_text)
    print("Success: the working copy now loads as JSON.")
    print("JSON structure:", type(ksu_data).__name__)
except json.JSONDecodeError as error:
    print("Another JSON issue:", error.msg)
    print("Position:", error.pos)
    print("Nearby text:", repr(
        repaired_text[max(0, error.pos - 60):error.pos + 60]
    ))

Success: the working copy now loads as JSON.
JSON structure: list


In [6]:
print("Number of records:", len(ksu_data))

if ksu_data:
    first_record = ksu_data[0]

    if isinstance(first_record, dict):
        print("\nField names:")
        for field in first_record:
            print("-", field)

    print("\nFirst record:")
    print(json.dumps(first_record, ensure_ascii=False, indent=2))
else:
    print("The dataset is empty.")

Number of records: 12703

Field names:
- Authors
- Article Title
- Source Title
- Document Type
- Author Keywords
- Abstract
- Affiliations
- DOI

First record:
{
  "Authors": "Al-Baadani, HH; Alharthi, AS; Abbas, NI; Qasem, AA; Saleh, A; Ibraheem, MA",
  "Article Title": "Effect of activated and inactivated Saccharomyces cerevisiae as alternative to antibiotic growth promoter on the performance and health of broilers infected with Clostridium perfringens",
  "Source Title": "ITALIAN JOURNAL OF ANIMAL SCIENCE",
  "Document Type": "Article",
  "Author Keywords": "S. cerevisiae; necrotic enteritis; performance; health; broilers",
  "Abstract": "The study aims to evaluate the addition of activated and inactivated Saccharomyces cerevisiae on the performance and health of broilers infected with Clostridium perfringens. 360 1-day-old male Ross 308 broilers were divided into 5 groups of 12 cages as experimental units (6 birds per cage) as follows: NC = negative control, basal diet; PC = posit

In [7]:
# This is individual publication data, and the file contains 12,703 records


# Create a folder for working data, separate from the original raw files.
interim_dir = project_dir / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)

# Choose a filename for the repaired JSON.
working_path = interim_dir / "ksu_publications_2025_repaired.json"

# Save the repaired text without changing the original downloaded file.
# UTF-8 preserves Arabic and English characters.
working_path.write_text(repaired_text, encoding="utf-8")

# Document the source and exactly what we repaired.
repair_note = {
    # The address used to download the original file.
    "source_url": url,

    # This is the year on the source file, not a verified year per paper.
    "source_file_year": 2025,

    # Count the records in the parsed JSON list.
    "record_count": len(ksu_data),

    # Store paths relative to the project so teammates can use them.
    "original_file": original_path.relative_to(project_dir).as_posix(),
    "working_file": working_path.relative_to(project_dir).as_posix(),

    # Record the zero-based character position of the repair.
    "repair_position": position,

    # Explain the change and its limits.
    "repair": (
        "Escaped a backslash followed by a literal tab to make valid JSON. "
        "Preserved both character values; did not reconstruct LaTeX."
    ),
}

# Save the repair note as a separate, readable JSON file.
note_path = interim_dir / "ksu_2025_repair_note.json"
note_path.write_text(
    json.dumps(repair_note, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Confirm which files were saved.
print("Saved working copy:", working_path.name)
print("Saved repair note:", note_path.name)

Saved working copy: ksu_publications_2025_repaired.json
Saved repair note: ksu_2025_repair_note.json


In [8]:
# Use separate variables so the 2025 data stays available.
url_2024 = (
    "https://"
    "data.ksu.edu.sa/sites/data.ksu.edu.sa/files/users/user976/"
    "KSU-DMO-OD-DATASET-SciResarch-Publications-2024.json"
)

# Download the file with a 90-second timeout.
with urlopen(url_2024, timeout=90) as response:
    raw_bytes_2024 = response.read()

# Save the original bytes before trying to parse the JSON.
original_path_2024 = raw_dir / "ksu_publications_2024_original.json"
original_path_2024.write_bytes(raw_bytes_2024)

print("Original saved:", original_path_2024.name)

# Decode the text and check whether it is valid JSON.
text_2024 = raw_bytes_2024.decode("utf-8-sig")

try:
    ksu_data_2024 = json.loads(text_2024)
    print("JSON structure:", type(ksu_data_2024).__name__)

    if isinstance(ksu_data_2024, list):
        print("Number of records:", len(ksu_data_2024))

        if ksu_data_2024:
            print("\nFirst record:")
            print(json.dumps(
                ksu_data_2024[0], ensure_ascii=False, indent=2
            ))
    else:
        print("Preview:", repr(ksu_data_2024)[:1500])

except json.JSONDecodeError as error:
    # Report any issue without changing the downloaded content.
    print("JSON issue:", error.msg)
    print("Line:", error.lineno, "| Column:", error.colno)
    print("Position:", error.pos)
    print("Nearby text:", repr(
        text_2024[max(0, error.pos - 100):error.pos + 100]
    ))

Original saved: ksu_publications_2024_original.json
JSON structure: list
Number of records: 16134

First record:
{
  "Authors": "Verma, M; Meena, RK; Khan, MI; Khatib, JM",
  "Article Title": "Analysis of the effect of curing and mixing periods on mechanical properties of the geopolymer composite",
  "Source Title": "MATERIALS SCIENCE-POLAND",
  "Document Type": "Article",
  "Author Keywords": "Geopolymer concrete; Alkali-silica reaction; Sustainable Construction; Durability; Heat curing",
  "Affiliations": "GLA University; National Institute of Technology (NIT System); National Institute of Technology Delhi; King Saud University; University of Wolverhampton"
}


In [9]:
# Keep the 2023 data separate from the other years.
url_2023 = (
    "https://"
    "data.ksu.edu.sa/sites/data.ksu.edu.sa/files/users/user976/"
    "KSU-DMO-OD-DATASET-SciResarch-Publications-2023.json"
)

# Download the original file.
with urlopen(url_2023, timeout=90) as response:
    raw_bytes_2023 = response.read()

# Preserve the downloaded bytes unchanged.
original_path_2023 = raw_dir / "ksu_publications_2023_original.json"
original_path_2023.write_bytes(raw_bytes_2023)

print("Original saved:", original_path_2023.name)

# Check whether the file contains valid JSON.
text_2023 = raw_bytes_2023.decode("utf-8-sig")

try:
    ksu_data_2023 = json.loads(text_2023)
    print("JSON structure:", type(ksu_data_2023).__name__)

    if isinstance(ksu_data_2023, list):
        print("Number of records:", len(ksu_data_2023))

        if ksu_data_2023:
            print("\nFirst record:")
            print(json.dumps(
                ksu_data_2023[0], ensure_ascii=False, indent=2
            ))
    else:
        print("Preview:", repr(ksu_data_2023)[:1500])

except json.JSONDecodeError as error:
    # Report problems without modifying the original file.
    print("JSON issue:", error.msg)
    print("Line:", error.lineno, "| Column:", error.colno)
    print("Position:", error.pos)
    print("Nearby text:", repr(
        text_2023[max(0, error.pos - 100):error.pos + 100]
    ))

Original saved: ksu_publications_2023_original.json
JSON structure: list
Number of records: 12346

First record:
{
  "Authors": "Abbas, H; Tao, W; Khan, G; Alrefaei, AF; Iqbal, J; Albeshr, MF; Kulsoom, I",
  "Article Title": "Multilayer perceptron and Markov Chain analysis based hybrid-approach for predicting land use land cover change dynamics with Sentinel-2 imagery",
  "Source Title": "GEOCARTO INTERNATIONAL",
  "Document Type": "Article",
  "Author Keywords": "Land use land cover; future prediction; multilayer perceptron; Markov Chain analysis; Sentinel 2",
  "Abstract": "As urbanization accelerates, the degree of human impact on land use is increasing. land use land cover change (LULC) is acknowledged as crucial factor in environmental change. The best way to understand historical land use patterns, changes, drivers, and developments is through a rigorous assessment of LULC changes. In this study, we aim to identify LULC changes from 2015 to 2022, and predict changes for 2030. Sen

In [10]:
# Use the parsed datasets; the 2025 data uses our repaired working copy.
datasets = {
    2023: ksu_data_2023,
    2024: ksu_data_2024,
    2025: ksu_data,
}

availability_report = []

for year, records in datasets.items():
    # Stop if any record has an unexpected structure.
    assert all(isinstance(record, dict) for record in records)

    print(f"\n--- Source file year: {year} ---")
    print("Total records:", len(records))

    for field in ["DOI", "Abstract"]:
        # Count absent fields separately from fields with empty values.
        absent = sum(field not in record for record in records)

        empty = sum(
            field in record
            and (
                record[field] is None
                or (
                    isinstance(record[field], str)
                    and not record[field].strip()
                )
            )
            for record in records
        )

        populated = len(records) - absent - empty

        availability_report.append({
            "source_file_year": year,
            "field": field,
            "total_records": len(records),
            "absent": absent,
            "empty": empty,
            "populated": populated,
        })

        print(
            f"{field}: {populated:,} populated | "
            f"{absent:,} absent | {empty:,} empty"
        )

# Save the findings for your source notes and enrichment planning.
report_path = interim_dir / "ksu_field_availability.json"
report_path.write_text(
    json.dumps(availability_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nReport saved:", report_path.name)


--- Source file year: 2023 ---
Total records: 12346
DOI: 2,986 populated | 0 absent | 9,360 empty
Abstract: 12,339 populated | 0 absent | 7 empty

--- Source file year: 2024 ---
Total records: 16134
DOI: 0 populated | 16,134 absent | 0 empty
Abstract: 0 populated | 16,134 absent | 0 empty

--- Source file year: 2025 ---
Total records: 12703
DOI: 12,703 populated | 0 absent | 0 empty
Abstract: 12,703 populated | 0 absent | 0 empty

Report saved: ksu_field_availability.json


In [11]:

#make an initial technology filter using titles and keywords, 
# which are available across all three years.
#  This first pass includes research applying technology in other fields, such as AI in medicine or agriculture.

import re

# Starting vocabulary: we can expand it after reviewing the results.
tech_terms = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
    "neural network",
    "neural networks",
    "multilayer perceptron",
    "random forest",
    "computer vision",
    "natural language processing",
    "large language model",
    "large language models",
    "generative ai",
    "cybersecurity",
    "cyber security",
    "information security",
    "intrusion detection",
    "cryptography",
    "blockchain",
    "software engineering",
    "internet of things",
    "iot",
    "cloud computing",
    "edge computing",
    "wireless network",
    "wireless networks",
    "robotics",
    "robot",
    "robots",
    "data mining",
    "big data",
    "computer science",
]

# Match complete phrases, ignoring capitalization.
patterns = {
    term: re.compile(r"\b" + re.escape(term) + r"\b", re.IGNORECASE)
    for term in tech_terms
}

tech_candidates = {}

for year, records in datasets.items():
    selected = []

    for row_index, record in enumerate(records):
        # Use the same fields for every year.
        searchable_text = " ".join(
            str(record.get(field) or "")
            for field in ["Article Title", "Author Keywords"]
        )

        matched_terms = [
            term
            for term, pattern in patterns.items()
            if pattern.search(searchable_text)
        ]

        if matched_terms:
            # Copy the record so the original dataset remains unchanged.
            candidate = record.copy()
            candidate["source_file_year"] = year
            candidate["source_row_index"] = row_index
            candidate["tech_matched_terms"] = matched_terms
            selected.append(candidate)

    tech_candidates[year] = selected

    print(f"\n{year}: {len(selected):,} candidates / {len(records):,} records")

    # Show a few titles so we can review the filter.
    for candidate in selected[:5]:
        print("-", candidate.get("Article Title"))
        print("  Matched:", ", ".join(candidate["tech_matched_terms"]))


2023: 938 candidates / 12,346 records
- Multilayer perceptron and Markov Chain analysis based hybrid-approach for predicting land use land cover change dynamics with Sentinel-2 imagery
  Matched: multilayer perceptron
- Enhancing deep learning techniques for the diagnosis of the novel coronavirus (COVID-19) using X-ray images
  Matched: artificial intelligence, deep learning, neural networks
- A Review of the Scope, Future, and Effectiveness of Using Artificial Intelligence in Cardiac Rehabilitation: A Call to Action for the Kingdom of Saudi Arabia
  Matched: artificial intelligence
- Modelling of land use and land cover changes and prediction using CA-Markov and Random Forest
  Matched: random forest
- County-level corn yield prediction using supervised machine learning
  Matched: machine learning

2024: 1,306 candidates / 16,134 records
- 3-D Trajectory Optimization and Communication Resources Allocation in UAV-Assisted IoT Networks for Sustainable Industry 5.0
  Matched: iot
- 6G-E

In [12]:

# The filter selected 3,533 candidate records across the three files. 
# The examples fit our broad scope, including technology applied to healthcare and agriculture. 
# These counts are before deduplication, and we still need to review some matches and excluded records.


# Save each year's candidates separately.
# Original downloaded files remain unchanged.
for year, candidates in tech_candidates.items():
    output_path = interim_dir / f"ksu_tech_candidates_{year}.json"

    output_path.write_text(
        json.dumps(candidates, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"Saved {len(candidates):,} candidates: {output_path.name}")

# Document how the initial filter selected records.
filter_note = {
    "status": "Initial candidates; manual review pending",
    "scope": "Technology research, including applications in other fields",
    "fields_searched": ["Article Title", "Author Keywords"],
    "matching_method": "Case-insensitive phrases with word boundaries",
    "terms": tech_terms,
    "counts_by_source_file_year": {
        str(year): len(candidates)
        for year, candidates in tech_candidates.items()
    },
    "limitations": [
        "Keyword matching may include irrelevant or miss relevant papers.",
        "Source file year is not a verified publication year.",
        "Records have not yet been deduplicated.",
    ],
    "enrichment_reminder": (
        "All 2024 source records lack DOI and Abstract fields. "
        "2023 also has missing values. Recheck missing fields among "
        "selected candidates before API enrichment."
    ),
}

# Save the filter documentation alongside the candidate files.
note_path = interim_dir / "ksu_tech_filter_note.json"
note_path.write_text(
    json.dumps(filter_note, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Saved filter documentation:", note_path.name)

Saved 938 candidates: ksu_tech_candidates_2023.json
Saved 1,306 candidates: ksu_tech_candidates_2024.json
Saved 1,289 candidates: ksu_tech_candidates_2025.json
Saved filter documentation: ksu_tech_filter_note.json


In [13]:
import random

# A fixed seed makes the sample reproducible.
rng = random.Random(42)

for year, candidates in tech_candidates.items():
    print(f"\n--- {year}: selected papers for review ---")

    sample = rng.sample(candidates, min(5, len(candidates)))

    for number, record in enumerate(sample, start=1):
        print(f"\n{number}. {record.get('Article Title', '')}")
        print("Keywords:", record.get("Author Keywords", ""))
        print("Matched terms:", ", ".join(record["tech_matched_terms"]))
        print("Original row index:", record["source_row_index"])


--- 2023: selected papers for review ---

1. A Novel Hybrid MPPT Approach for Solar PV Systems Using Particle-Swarm-Optimization-Trained Machine Learning and Flying Squirrel Search Optimization
Keywords: DC-DC converter; MPPT algorithm; solar photovoltaic system
Matched terms: machine learning
Original row index: 10091

2. Smart Deep Learning Model to Recognize PCM Optimization Performance on Solar Cooling System
Keywords: PCM optimization; photovoltaic systems; renewable energy
Matched terms: deep learning
Original row index: 1856

3. Solid oxide fuel cell energy system with absorption-ejection refrigeration optimized using a neural network with multiple objectives
Keywords: Solid oxide fuel cell; Absorption-ejection refrigeration; Neural network Genetic algorithm Energy efficiency
Matched terms: neural network
Original row index: 381

4. Optimal Fuzzy Wavelet Neural Network Based Road Damage Detection
Keywords: Flooding; road damage; machine learning; parameter tuning; computer visi

In [19]:

# Rebuild the candidate lists with a more precise matching rule.
tech_candidates = {}

for year, records in datasets.items():
    selected = []

    for row_index, record in enumerate(records):
        searchable_text = " ".join(
            str(record.get(field) or "")
            for field in ["Article Title", "Author Keywords"]
        )

        # Avoid confusing the eye-health condition with computing research.
        text_for_matching = re.sub(
            r"\bcomputer\s+vision\s+syndrome\b",
            " ",
            searchable_text,
            flags=re.IGNORECASE,
        )

        matched_terms = [
            term
            for term, pattern in patterns.items()
            if pattern.search(text_for_matching)
        ]

        if matched_terms:
            candidate = record.copy()
            candidate["source_file_year"] = year
            candidate["source_row_index"] = row_index
            candidate["tech_matched_terms"] = matched_terms
            selected.append(candidate)

    tech_candidates[year] = selected
    print(f"{year}: {len(selected):,} candidates")

# Update the saved candidate files.
for year, candidates in tech_candidates.items():
    output_path = interim_dir / f"ksu_tech_candidates_{year}.json"
    output_path.write_text(
        json.dumps(candidates, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

# Document the revised rule and counts.
filter_note["excluded_phrase_before_matching"] = "computer vision syndrome"
filter_note["counts_by_source_file_year"] = {
    str(year): len(candidates)
    for year, candidates in tech_candidates.items()
}

(interim_dir / "ksu_tech_filter_note.json").write_text(
    json.dumps(filter_note, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Updated candidate files and filter notes.")

2023: 940 candidates
2024: 1,306 candidates
2025: 1,289 candidates
Updated candidate files and filter notes.


In [15]:
# Use a fixed seed so this review sample is reproducible.
rng = random.Random(42)

for year, records in datasets.items():
    # Identify the original rows already selected by the filter.
    selected_indices = {
        record["source_row_index"]
        for record in tech_candidates[year]
    }

    excluded = [
        (index, record)
        for index, record in enumerate(records)
        if index not in selected_indices
    ]

    print(f"\n--- {year}: excluded papers for review ---")

    for index, record in rng.sample(excluded, min(5, len(excluded))):
        print("\nOriginal row index:", index)
        print("Title:", record.get("Article Title", ""))
        print("Keywords:", record.get("Author Keywords", ""))


--- 2023: excluded papers for review ---

Original row index: 11212
Title: Estimation of electrostatic and covalent contributions to the enthalpy of H-bond formation in H-complexes of 1,2,3-benzotriazole with proton-acceptor molecules by IR spectroscopy and DFT calculations
Keywords: H-bond; 1,2,3-benzotriazole; DFT; AIM; ELF; NCI; RDG

Original row index: 1945
Title: Cu2ZnGeSe4 single crystals: Growth, structure and temperature dependence of band gap
Keywords: Single crystal growth; Crystal structure; Band gap; Gas chemical method; Semiconducting quaternary alloys

Original row index: 439
Title: Impact of varied fog collector designs on fog and rainwater harvesting under fluctuating wind speed and direction
Keywords: Fog; Rainwater; Mesh design; ANN modeling; Arid region

Original row index: 4817
Title: A review on recent advances in covalent organic frameworks-based membranes: Synthesis, modification, and applications in liquid phase separation
Keywords: Covalent-organic frameworks;

In [16]:

#found a likely missed match: the 2023 fog-collector paper lists 
# “ANN modeling”, commonly used for artificial neural network modeling. 
# Let’s inspect its abstract before adding that abbreviation to the filter.


# Next: inspect the ANN paper.

# Inspect the original record without changing it.
record = datasets[2023][439]

print("Title:", record.get("Article Title", ""))
print("\nKeywords:", record.get("Author Keywords", ""))
print("\nAbstract:", record.get("Abstract") or "No abstract available")

Title: Impact of varied fog collector designs on fog and rainwater harvesting under fluctuating wind speed and direction

Keywords: Fog; Rainwater; Mesh design; ANN modeling; Arid region

Abstract: This study explores innovative approaches to fog water collection by introducing a novel cylindrical mesh design, deviating from the traditional flat mesh. Motivated by the need to address the impact of variable wind directions on collection efficiency, field and laboratory experiments were conducted. Field experiments incorporated both square and cylindrical mesh designs, strategically placed to accommodate natural wind variations. Simultaneously, laboratory experiments considered two wind speeds (7 and 9 km/h) for both designs. Results indicated higher water collection quantities at 7 km/h compared to 9 km/h for both designs. The study emphasizes the critical role of wind direction stability. In regions with consistent wind directions during fog and rain events, the square design proves mo

In [17]:

#Confirmed: this paper uses artificial neural networks, so it fits our scope of computing and AI applications.
#Next, add “ANN modeling” and “ANN modelling” to the filter. 
# These phrases are more specific than matching ANN alone.


# Add both spellings without creating duplicate entries.
for term in ["ann modeling", "ann modelling"]:
    if term not in tech_terms:
        tech_terms.append(term)

# Rebuild the matching patterns with the updated vocabulary.
patterns = {
    term: re.compile(r"\b" + re.escape(term) + r"\b", re.IGNORECASE)
    for term in tech_terms
}

# Keep the saved documentation aligned with the updated vocabulary.
filter_note["terms"] = tech_terms.copy()
filter_note["review_finding"] = (
    "The 2023 record at row index 439 was missed. Its abstract confirms "
    "artificial neural networks, and its keywords contain ANN modeling. "
    "Added ANN modeling and ANN modelling to the vocabulary."
)

print("Added ANN modeling and ANN modelling.")

Added ANN modeling and ANN modelling.


In [20]:

candidate_availability = []

for year, candidates in tech_candidates.items():
    print(f"\n--- {year}: {len(candidates):,} tech candidates ---")

    for field in ["DOI", "Abstract"]:
        # Count absent, null, or whitespace-only values as missing.
        missing = sum(
            record.get(field) is None
            or (
                isinstance(record.get(field), str)
                and not record[field].strip()
            )
            for record in candidates
        )

        candidate_availability.append({
            "source_file_year": year,
            "field": field,
            "candidate_count": len(candidates),
            "missing_count": missing,
        })

        print(f"{field}: {missing:,} missing")

# Save the findings for API enrichment planning.
report_path = interim_dir / "ksu_tech_enrichment_needs.json"
report_path.write_text(
    json.dumps(candidate_availability, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nSaved:", report_path.name)


--- 2023: 940 tech candidates ---
DOI: 748 missing
Abstract: 0 missing

--- 2024: 1,306 tech candidates ---
DOI: 1,306 missing
Abstract: 1,306 missing

--- 2025: 1,289 tech candidates ---
DOI: 0 missing
Abstract: 0 missing

Saved: ksu_tech_enrichment_needs.json


In [ ]:
enrichment_queue = []

for year, candidates in tech_candidates.items():
    for record in candidates:
        # Identify which fields need enrichment.
        missing_fields = [
            field
            for field in ["DOI", "Abstract"]
            if record.get(field) is None
            or (
                isinstance(record.get(field), str)
                and not record[field].strip()
            )
        ]

        if missing_fields:
            enrichment_queue.append({
                "source_file_year": year,
                "source_row_index": record["source_row_index"],
                "title": record.get("Article Title", ""),
                "authors": record.get("Authors", ""),
                "journal": record.get("Source Title", ""),
                "existing_doi": record.get("DOI"),
                "missing_fields": missing_fields,
                "status": "pending",
            })

# Save lookup requests without modifying the candidate datasets.
queue_path = interim_dir / "ksu_enrichment_queue.json"
queue_path.write_text(
    json.dumps(enrichment_queue, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Records queued:", len(enrichment_queue))
print("Saved:", queue_path.name)


Records queued: 2054
Saved: ksu_enrichment_queue.json
